# Advanced Example 6: multi-arm X-SHOOTER classification

This is an advanced optional example, not a required step after the core Examples 1–5 path.

This notebook shows how to work with several X-SHOOTER arms together. It uses bundled UVB, VIS, and NIR spectra of the same target and keeps the arms as separate `SpectrumSegment` objects inside one `SpectrumCollection`.

The goal is classification and consistency checking. UVB Balmer wings, VIS Hα/Ca-triplet regions, and NIR Paschen/Brackett/CO windows can all be useful, but each arm has its own resolution, telluric exposure, flux-calibration behaviour, and residual risks.

## What this example teaches

- how to keep UVB, VIS, and NIR arms as separate `SpectrumSegment` objects;
- how to inspect broad classification evidence across atomic lines and molecular bands;
- how to fit only retained windows while leaving fragmented/telluric-sensitive regions diagnostic-only.

## Requirements

The default inspection path uses bundled X-SHOOTER arms only. The optional multi-arm fit requires PHOENIX and an explicit exploratory override if the audit blocks interpretation.

## Expected outputs

Per-arm summaries, all-arm diagnostic-window plots, a retained-window provenance report, and optional UVB/VIS fit plots.


Note: A multi-arm diagnostic view is not a hidden arm-alignment or SED-calibration correction; Spyctres preserves the arm provenance and only fits the retained windows.


## What this example adds

Examples 1–4 mostly teach single-spectrum or UVB/Balmer reasoning. Here we use spectra from the same target as in examples 2,3 and 4, and add the multi-arm pattern:

1. read UVB, VIS, and NIR products with the same reader;
2. keep the arms separate in a `SpectrumCollection` rather than merging them onto one grid;
3. inspect per-arm provenance, mask fraction, and resolution/LSF metadata;
4. ask Spyctres which diagnostic windows overlap the combined wavelength range;
5. build an explicit mask bundle where `valid_mask=True` means usable;
6. optionally run a bounded multi-arm PHOENIX fit over selected diagnostic windows;
7. interpret the result as a consistency check unless the stronger reviewed-analysis checks also pass.

We want to look at Hα, Paschen/Brackett hydrogen lines, Ca II triplet, TiO, and CO as classification evidence. Example 4 remains the stricter reviewed-analysis UVB Balmer-wing workflow, and we will not repeat that analysis here.


## 0. Controls

The default notebook run is PHOENIX-free so a new user can inspect the multi-arm data quickly. Set `RUN_MULTIARM_FIT = True` after reviewing the selected windows and masks.


In [ ]:
import numpy as np
import Spyctres as sp

PHOENIX_DIR = None
# Leave this False for the normal teaching run. Set it True only after
# reviewing the masks/windows and confirming PHOENIX is configured.
RUN_MULTIARM_FIT = True
SHOW_PROGRESS = True

# If the audit blocks a fit, this explicit switch records that the run is
# exploratory. Keep it False unless you are deliberately doing a diagnostic run.
ALLOW_EXPLORATORY_FIT = True
OVERRIDE_REASON = "Example 6 multi-arm classification tutorial; not final analysis."

reader = "xshooter_merge1d"


## 1. Read the three bundled X-SHOOTER arms

The reader describes the reduced data product. `xshooter_merge1d` reads a merged 1-D X-SHOOTER FITS image product and preserves the arm-specific metadata it can infer from the header.

You can discover readers with `sp.list_readers()` and inspect a reader with `sp.get_reader_info("xshooter_merge1d")`.


In [ ]:
arm_files = {
    "UVB": "TOO_Gaia21ccu_SCI_SLIT_FLUX_MERGE1D_UVB.fits",
    "VIS": "TOO_Gaia21ccu_SCI_SLIT_FLUX_MERGE1D_VIS_TELL_CORR.fits",
    "NIR": "TOO_Gaia21ccu_SCI_SLIT_FLUX_MERGE1D_NIR_TELL_CORR.fits",
}

segments = []
for arm_label, filename in arm_files.items():
    segment = sp.read_spectrum(sp.example_data_path(filename), reader=reader)
    segment.meta["example_arm_label"] = arm_label
    segments.append(segment)

multiarm = sp.SpectrumCollection(
    segments,
    name="example6_xshooter_multiarm",
    meta={"workflow": "example6_multiarm_classification"},
)

print(multiarm.summary())


## 2. Check per-arm provenance before fitting

A multi-arm object is not one homogeneous spectrum. The UVB, VIS, and NIR arms can have different wavelength coverage, resolution, masks, and telluric behaviour. Spyctres keeps those differences visible instead of silently flattening them into one array.


In [ ]:
for segment in multiarm.segments:
    summary = segment.summary()
    resolution = summary.get("resolution") or {}
    print(segment.meta.get("example_arm_label"), segment.name)
    print(
        "  coverage: "
        f"{summary['wavelength_range_A'][0]:.1f}-{summary['wavelength_range_A'][1]:.1f} Å"
    )
    print(
        "  conventions: "
        f"medium={summary.get('wave_medium')}, "
        f"observer_frame={summary.get('observer_frame')}, "
        f"stellar_rest={summary.get('stellar_rest_status')}"
    )
    print(
        "  resolution/LSF: "
        f"R={resolution.get('value')} from {resolution.get('source')}"
    )


## 3. Plot the arms before choosing windows

The first plot shows the raw arm fluxes. The audit plot uses local display scaling and mask panels to make problematic regions easier to see. These plots are diagnostic displays only; Spyctres is not automatically re-scaling or re-aligning the arms.

The arms may not look perfectly aligned in a full-spectrum display. That is not automatically a problem for the line-based classification shown here, because we inspect local windows and later fit each arm with its own mask, uncertainty, and resolution metadata. It would matter much more for an SED/flux-calibration analysis, extinction estimate, global continuum fit, or any diagnostic window that sits directly on an arm join or poor edge region.


In [ ]:
sp.plot_spectrum(
    multiarm,
    title="Example 6: loaded X-SHOOTER UVB/VIS/NIR arms",
    figsize=(14.0, 4.2),
)

sp.plot_spectrum_audit(
    multiarm,
    title="Example 6: generic multi-arm audit view before fitting",
    figsize=(14.0, 8.0),
)


## 4. Ask Spyctres for diagnostic windows across the full coverage

The ranked list can include hot-star hydrogen/helium/metal windows, cool-star molecular windows, and near-IR checks. Not every listed window should be fitted for every target. For this target, we already know from the previous examples that the UVB Balmer wings carry the main hot-star information, while VIS and NIR features are valuable consistency checks.

In the next section we will consider which windows support a hot/intermediate classification, and which windows would argue for a cooler star if they were strong?


In [ ]:
windows = sp.select_diagnostic_windows(multiarm, max_windows=30)
print(windows.summary_text(max_rows=30))

sp.plot_diagnostic_windows(
    multiarm,
    selection=windows,
    title="Example 6: diagnostic windows across UVB/VIS/NIR",
    figsize=(14.0, 4.2),
)


## 5. Build reviewed masks

Here archive/product bad regions are excluded when available. Tellurics are warning overlays rather than automatic exclusions because these VIS/NIR files are already telluric-corrected products, and remaining telluric residuals should be inspected rather than silently hidden. For a different dataset you might choose `tellurics="mask"` after checking the frame and product provenance.


In [ ]:
reviewed_mask = sp.build_mask(
    multiarm,
    archive="mask",
    tellurics="warn",
    dibs=False,
)
print(reviewed_mask.summary_text())

sp.plot_spectrum_audit(
    multiarm,
    diagnostic_selection=windows,
    warning_regions=reviewed_mask.warning_regions,
    title="Example 6: multi-arm audit with windows and warning regions",
    figsize=(14.0, 8.0),
)


## 6. Choose hot-vs-cool classification windows

The automatic ranked list is useful, but classification is a scientific question. Here we choose diagnostic windows that answer two complementary questions:

1. **Does the spectrum look hot/intermediate?** Look for Balmer, Paschen, Brackett, He I, and Mg II behaviour.
2. **Does the spectrum look cool?** Look for stronger Ca/Na metal lines and molecular TiO/CO structure.

We also check whether each window is usable enough to fit. A window can be useful for visual diagnosis but still be a poor default fit window if it is near an arm edge, heavily fragmented by masks/tellurics, or model-sensitive. The rule of thumb here is: inspect broadly, fit narrowly.


In [ ]:
# These groups are tutorial choices, not hidden Spyctres defaults.
# They keep the classification question visible: hot/intermediate evidence
# versus cool-star evidence.
hot_window_ids = [
    "h_delta",
    "h_gamma",
    "h_beta",
    "h_alpha",
    "he_i_4471",
    "mg_ii_4481",
    "he_i_5876",
    "paschen_beta",
    "br_gamma",
]

cool_window_ids = [
    "ca_hk_h_epsilon",
    "ch_g_band",
    "mg_i_b",
    "na_i_d",
    "ca_ii_triplet_paschen",
    "tio_7050",
    "na_i_8200",
    "tio_red_bands",
    "na_i_kband",
    "ca_i_kband",
    "co_23um_bandhead",
]

# These are useful to inspect, but in this dataset they are especially
# telluric-, continuum-, or edge-sensitive. We keep them visible as diagnostics
# rather than making them default fit constraints.
context_only_window_ids = [
    "paschen_gamma_delta",
    "brackett_h_band",
]

# Extra plot markers for broad NIR and cool-star windows. These are
# annotations only: they help orient your eye, but they are not masks,
# corrections, or automatic physical identifications.
nir_hydrogen_markers = [
    ("Paδ", 10049.37),
    ("Paγ", 10938.09),
    ("Paβ", 12818.08),
    ("Brγ", 21661.0),
]

cool_star_markers = [
    ("TiO 7050", 7050.0),
    ("Na I 2.21 μm", 22062.0),
    ("Ca I 2.26 μm", 22630.0),
    ("CO 2-0", 22935.0),
]

classification_window_groups = [
    ("hot/intermediate evidence", hot_window_ids),
    ("cool-star checks", cool_window_ids),
    ("NIR context only", context_only_window_ids),
]

edge_margin_A = 150.0
min_usable_fraction_for_fit = 0.65
min_contiguous_fraction_for_fit = 0.30
conservative_fit_preference = {
    "h_delta",
    "h_gamma",
    "h_beta",
    "h_alpha",
}


In [ ]:
classification_windows_by_id = {}
classification_rows = []

for group_label, group_ids in classification_window_groups:
    for item in windows.select_by_id(
        group_ids,
        require_all=False,
        include_rejected=True,
    ):
        if item["id"] in classification_windows_by_id:
            continue

        near_edge = False
        contributing_arms = []
        for contribution in item.get("segment_contributions", []):
            if contribution.get("n_pixels", 0) <= 0:
                continue
            segment = multiarm.segments[contribution["segment_index"]]
            arm_label = segment.meta.get("example_arm_label", segment.name)
            contributing_arms.append(str(arm_label))

            # Compare the operational catalog window with this arm's actual
            # wavelength coverage. Near-edge windows are fine to inspect but
            # risky as default fit constraints.
            arm_lo = float(np.nanmin(segment.wave))
            arm_hi = float(np.nanmax(segment.wave))
            op_lo, op_hi = contribution.get("operational_region_A", item["region_A"])
            if op_lo <= arm_lo + edge_margin_A or op_hi >= arm_hi - edge_margin_A:
                near_edge = True

        usable_fraction = float(item.get("usable_fraction", 0.0) or 0.0)
        contiguous_fraction = float(
            item.get("largest_contiguous_usable_fraction", 0.0) or 0.0
        )
        fragmented = contiguous_fraction < min_contiguous_fraction_for_fit
        low_usable = usable_fraction < min_usable_fraction_for_fit
        preferred_for_fit = item["id"] in conservative_fit_preference
        fit_candidate = preferred_for_fit and not near_edge and not fragmented and not low_usable

        if fit_candidate:
            decision = "fit candidate"
        elif near_edge:
            decision = "diagnostic only: near arm edge/join"
        elif fragmented:
            decision = "diagnostic only: fragmented usable pixels"
        elif low_usable:
            decision = "diagnostic only: low usable fraction"
        else:
            decision = "diagnostic only: classification check"

        row = {
            "id": item["id"],
            "label": item["label"],
            "group": group_label,
            "region_A": item["region_A"],
            "features": item.get("features", []),
            "arms": sorted(set(contributing_arms)),
            "usable_fraction": usable_fraction,
            "contiguous_fraction": contiguous_fraction,
            "risk_tags": item.get("risk_tags", []),
            "decision": decision,
            "fit_candidate": fit_candidate,
        }
        classification_windows_by_id[item["id"]] = item
        classification_rows.append(row)


In [ ]:
print("Hot-vs-cool diagnostic windows:")
for row in classification_rows:
    print(
        f"- {row['id']:24s} {row['label']:28s} "
        f"arms={','.join(row['arms']) or 'none':8s} "
        f"usable={row['usable_fraction']:.2f} "
        f"contiguous={row['contiguous_fraction']:.2f} "
        f"-> {row['decision']}"
    )
    if row["risk_tags"]:
        print("    risks:", ", ".join(row["risk_tags"]))

fit_window_ids = [row["id"] for row in classification_rows if row["fit_candidate"]]
fit_windows = sorted(
    windows.select_by_id(
        fit_window_ids,
        require_all=False,
        include_rejected=True,
    ),
    key=lambda item: float(item["region_A"][0]),
)
print("\nConservative windows for the optional joint fit:")
for item in fit_windows:
    print(f"  {item['id']}: {item['label']} {item['region_A']}")
print(
    "These are conservative fit candidates. Other plotted VIS/NIR windows "
    "remain classification checks unless their masks, tellurics, edge "
    "distance, and model suitability have been reviewed."
)


## 7. Plot the selected diagnostic windows before fitting

These plots are observed-spectrum diagnostics only. They help the user decide which lines actually look informative before running an expensive model fit. Orange/grey line markers are orientation aids, not automatic masks or automatic feature identifications.

The hot/intermediate plot asks whether hydrogen/helium diagnostics support a hot classification. The cool-star plot asks the opposite question: do we see strong Ca/Na/TiO/CO features that would push us toward a cooler star or a composite/contaminated interpretation?


In [ ]:
# Concatenate the arms only for this display plot. The SpectrumCollection
# still keeps the arms separate for fitting, masks, resolution, and provenance.
display_wave = np.concatenate([segment.wave for segment in multiarm.segments])
display_flux = np.concatenate([segment.flux for segment in multiarm.segments])
display_valid = np.concatenate(reviewed_mask.valid_masks_by_segment)
display_order = np.argsort(display_wave)

hot_windows = windows.select_by_id(
    hot_window_ids,
    require_all=False,
    include_rejected=True,
)
cool_windows = windows.select_by_id(
    cool_window_ids,
    require_all=False,
    include_rejected=True,
)
context_windows = windows.select_by_id(
    context_only_window_ids,
    require_all=False,
    include_rejected=True,
)

sp.plot_spectrum_line_windows(
    display_wave[display_order],
    display_flux[display_order],
    hot_windows,
    valid_mask=display_valid[display_order],
    line_groups=("balmer", "hei", "mgii", nir_hydrogen_markers),
    title="Example 6: hot/intermediate-star diagnostic windows",
    ncols=3,
    figsize_per_panel=(5.8, 3.1),
    footer=(
        "Hydrogen, He I, and Mg II windows are positive evidence for a hot or "
        "intermediate classification, but telluric/edge-sensitive NIR windows "
        "should be treated as checks, not automatic corrections."
    ),
)

sp.plot_spectrum_line_windows(
    display_wave[display_order],
    display_flux[display_order],
    cool_windows,
    valid_mask=display_valid[display_order],
    line_groups=("caii", "nai", cool_star_markers),
    title="Example 6: cool-star diagnostic windows",
    ncols=3,
    figsize_per_panel=(5.8, 3.1),
    footer=(
        "Strong Ca/Na/TiO/CO structure would support a cooler or composite "
        "interpretation. Weak or absent cool-star features are useful negative "
        "classification evidence."
    ),
)

if context_windows:
    sp.plot_spectrum_line_windows(
        display_wave[display_order],
        display_flux[display_order],
        context_windows,
        valid_mask=display_valid[display_order],
        line_groups=(nir_hydrogen_markers,),
        title="Example 6: NIR context-only windows",
        ncols=2,
        figsize_per_panel=(6.8, 3.1),
        footer=(
            "These windows are useful context, but this dataset marks them as "
            "edge/telluric/fragmentation-sensitive rather than default fit windows."
        ),
    )


## 8. Audit the exact multi-arm fit plan

Because we changed the region list ourselves, we ask for a setup without an embedded readiness claim and then audit the exact regions separately. This avoids confusing a setup summary for one window set with an audit of another window set.


In [ ]:
# Convert the inspected window list into the exact object used by the fit.
# The helper keeps the workflow generic: it checks every requested window
# against every loaded segment, preserves the original segment metadata, and
# records which arms/windows are fit-ready versus diagnostic-only context.
fit_selection = sp.build_fit_collection_from_windows(
    multiarm,
    windows,
    window_ids=fit_window_ids,
    valid_masks=reviewed_mask.valid_masks_by_segment,
    min_usable_fraction=min_usable_fraction_for_fit,
    min_contiguous_fraction=min_contiguous_fraction_for_fit,
    name="example6_xshooter_retained_fit_arms",
)

print(fit_selection.summary_text())

# These three objects are aligned and can be passed directly to the fitter.
fit_collection = fit_selection.collection
fit_valid_masks_by_segment = fit_selection.valid_masks_by_segment
fit_regions = list(fit_selection.regions)

# Keep only the retained windows for model-line plots after the fit.
fit_windows = sorted(
    windows.select_by_id(
        fit_selection.retained_window_ids,
        require_all=False,
        include_rejected=True,
    ),
    key=lambda item: float(item["region_A"][0]),
)

fit_audit = sp.audit_spectrum_for_fit(
    fit_collection,
    regions=fit_regions,
    intent="quicklook_classification",
)
print(
    "multi-arm fit audit:",
    "ready=", fit_audit.get("fit_ready"),
    "fitted_pixels=", fit_audit.get("n_fit_candidate"),
    "blockers=", fit_audit.get("blockers_for_intent"),
)
print("outside fit-window fraction:", fit_audit.get("outside_fit_window_fraction"))
print(
    "rejected inside fit-window fraction:",
    fit_audit.get("rejected_inside_fit_window_fraction"),
)

multiarm_setup = sp.suggest_fit_setup(
    fit_collection,
    mode="quicklook",
    intent="quicklook_classification",
).with_regions(fit_regions).with_readiness(fit_audit)

print("Optional-fit setup:")
print(multiarm_setup.summary_text(include_hash=False))


## 9. Optional bounded multi-arm PHOENIX fit

This fit uses all selected regions together, while preserving per-arm masks and per-arm resolution metadata. If the audit is blocked, the notebook records an exploratory override rather than pretending the result is ready for science use.

Leave `RUN_MULTIARM_FIT = False` while you are reading the tutorial. Set it to `True` only after the plots and selected windows make sense.


In [ ]:
multiarm_result = None
if RUN_MULTIARM_FIT:
    setup_for_fit = multiarm_setup
    if fit_audit.get("fit_ready") is not True:
        if not ALLOW_EXPLORATORY_FIT:
            raise ValueError(
                "The multi-arm audit is blocked. Inspect the plots first or "
                "set ALLOW_EXPLORATORY_FIT=True with a clear OVERRIDE_REASON."
            )
        setup_for_fit = multiarm_setup.allow_exploratory(reason=OVERRIDE_REASON)

    multiarm_result = sp.fit_stellar_spectrum(
        fit_collection,
        model="phoenix",
        setup=setup_for_fit,
        valid_mask=fit_valid_masks_by_segment,
        phoenix_dir=PHOENIX_DIR,
        progress_callback=(
            (lambda event: print(f"[{event.elapsed_s:6.1f}s] {event}", flush=True))
            if SHOW_PROGRESS
            else None
        ),
    )
    print(multiarm_result.summary_text(include_hash=False, max_flags=8))
else:
    print("RUN_MULTIARM_FIT is False. The notebook has prepared the multi-arm fit plan only.")


## 10. Inspect the optional multi-arm result

If you ran the fit, inspect both the full referee plot and the line-window residual panels.


In [ ]:
if multiarm_result is not None:
    sp.plot_fit_referee(
        multiarm_result,
        layout="stacked",
        flux_ylim_mode="visible",
    )

    sp.plot_model_line_windows(
        multiarm_result,
        windows=fit_windows[:8],
        title="Example 6: multi-arm diagnostic windows",
        show_residuals=True,
        residual_kind="pull",
        ncols=2,
        figsize_per_panel=(7.2, 5.2),
    )
else:
    print("Run the optional multi-arm fit cell first to generate model-residual plots.")


## 11. What we can conclude from these diagnostics

A sensible multi-arm classification asks whether independent wavelength regions agree:

- UVB Balmer wings constrain hot-star Teff/log g strongly;
- Hα can reveal emission, activity, wind, or continuum/LSF problems;
- Paschen and Brackett lines are useful near-IR hot-star consistency checks;
- Ca II triplet, TiO, Na, Ca, and CO become especially important for cooler stars;
- large arm-to-arm disagreements are evidence to investigate, not a reason to add hidden arm-scaling corrections by default.

For the bundled spectrum for this particular target, the most useful positive classification evidence is still the UVB Balmer series, with VIS Hα acting as an additional consistency check. The NIR hydrogen, Brackett, and CO windows are worth inspecting, but in this particular reduction the usable pixels are fragmented enough that Spyctres keeps them diagnostic-only by default. The cool-star windows are mostly negative evidence here: if TiO, K-band Na/Ca, or CO were strong, we would worry about a much cooler star, composite light, contamination, or reduction/model mismatch.

The fact that the three arms are not perfectly aligned in raw flux is therefore not fatal for this notebook. We are not fitting a physical SED or deriving extinction from the arm-to-arm continuum. We are using local line windows, per-arm metadata, masks, and resolution information. Arm alignment would become a central problem only for global continuum/SED work or if a chosen diagnostic window straddled a poor arm edge or join.

Example 4 remains the stricter reviewed-analysis Balmer-wing workflow. Example 6 shows how UVB/VIS/NIR information can be assembled into a classification and consistency story without hiding calibration problems.


In [ ]:
if multiarm_result is not None:
    print(multiarm_result.quality_report_text())
else:
    print("Next manual step: set RUN_MULTIARM_FIT=True after checking the audit plots.")


## 12. Command-line reproduction

The command-line version follows the same default: inspect first, fit only when requested.


In [ ]:
print("python examples/example6_multiarm_classification.py --no-show")
print(
    "python examples/example6_multiarm_classification.py "
    "--run-fit "
    "--allow-exploratory-fit "
    "--override-reason 'multi-arm tutorial classification; not final analysis' "
    "--plot-dir /tmp/spyctres_example6_multiarm "
    "--no-show"
)


## What to try next

- Edit `fit_window_ids` for a cooler star and include TiO, Ca I, Na I, and CO windows.
- Compare UVB-only, UVB+VIS, and UVB+VIS+NIR setups before trusting a single all-arm result.
- Use Example 4 when the question is reviewed-analysis stellar parameters for Gaia21ccu.
- Use Example 5 when the question is how to triage many spectra efficiently.


In [ ]:
sp.describe_public_function("select_diagnostic_windows")
